In [1]:
import sys, pathlib, re, json, hashlib, datetime, math
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query
import pandas as pd

_BLOCKED = re.compile(r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|CREATE|TRUNCATE|EXEC|EXECUTE|GRANT|REVOKE|DENY|INTO|BACKUP|RESTORE)\b", re.I)

def _strip(sql):
    sql = re.sub(r"'([^']|'')'", "''", sql)
    sql = re.sub(r"--[^\n]*", "", sql)
    return re.sub(r"/\*.*?\*/", "", sql, flags=re.S)

def q(sql):
    hit = _BLOCKED.search(_strip(sql))
    if hit:
        raise PermissionError(f"'{hit.group(0).upper()}' is not allowed: the database is read-only.")
    return run_query(sql)

_DATE = re.compile(r"^\d{4}-\d{1,2}-\d{1,2}")
def _norm(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    if isinstance(v, (datetime.date, datetime.datetime, pd.Timestamp)):
        return pd.Timestamp(v).date().isoformat()
    if isinstance(v, str):
        v = v.strip()
        if _DATE.match(v):
            try:
                return pd.Timestamp(v).date().isoformat()
            except ValueError:
                pass
        return v
    try:
        return round(float(v), 2)
    except (TypeError, ValueError):
        return str(v)

def _fingerprint(df):
    rows = [[_norm(v) for v in row] for row in df.itertuples(index=False)]
    return hashlib.sha256(json.dumps(rows).encode()).hexdigest()

EXPECTED = { "s01": [ [ 8, 2 ], "0ba3b4c0210bb324982933cd308c7ff0bb3fbbea32fc6646d9e9e2a9bb14df70" ], "s02": [ [ 1, 3 ], "1512ef1d7254b2206516b1f4bb9dea659b456cab8b80402f5bf67de186fc7fff" ], "s03": [ [ 3, 4 ], "607a2a3b80c9fdc820855d34f48730b445d9437842569c3d7406962b2ec71409" ], "s04": [ [ 26, 2 ], "2fe50527877da0fd0064e1512aa038aa08109668d30b5e0083874edc03b773cf" ], "s05": [ [ 5, 2 ], "8d5b93be4960cbb6acd605b1fa741b3be8bbe6c44c5fb19b843daf38737e1ab1" ], "s06": [ [ 1, 1 ], "c3c32ce95ec239c2ac639584925dced87cfb17b11bee9b738bc7ed104c2ce931" ], "s07": [ [ 6, 2 ], "8450a870c23b5bfab2a420e0a10a1b534e113a7c4acd0be5b90060f8bebe4e2c" ], "s08": [ [ 12, 3 ], "27236391b69c4478be3e09fa08b89fe7c5acd46b03bb5a934c14e6e122bddb0d" ], "s09": [ [ 12, 3 ], "880e60a4b694da3e224b104c67139870b74671f6ef86f16b9d5ac46c2d81ad04" ], "s10": [ [ 3, 4 ], "9b15fac9c91f6a72150b0c1705f96f0e8281cbe86e49c56d49f37a3d885328ac" ] }
TASKS = { "s01": "Exploring the schema", "s02": "Filtering & NULLs", "s03": "CASE & conditional aggregation", "s04": "GROUP BY + HAVING", "s05": "JOIN across sources", "s06": "Anti-join", "s07": "UNION ALL", "s08": "CTE + window ranking", "s09": "Running total (window)", "s10": "Business question" }
RESULTS = {}

def answer(task, sql):
    shape, fp = EXPECTED[task]
    if not _strip(sql).strip():
        RESULTS[task] = None
        print(f"⏳ {task}: not attempted yet. Write your query between the triple quotes.")
        return
    try:
        df = q(sql)
    except Exception as err:
        RESULTS[task] = False
        print(f"❌ {task}: the query failed.\n {type(err).__name__}: {str(err)[:300]}")
        return
    display(df.head(15))
    ok = _fingerprint(df) == fp
    RESULTS[task] = ok
    if ok:
        print(f"✅ {task}: correct!")
        return
    print(f"❌ {task}: not quite. Expected {shape[0]} row(s) × {shape[1]} column(s); you returned {df.shape[0]} × {df.shape[1]}.")
    if df.shape == tuple(shape):
        print(" The shape is right, so check: column order, sort order, rounding, and filters.")

def sql_score():
    rows = []
    for task, title in TASKS.items():
        r = RESULTS.get(task)
        rows.append([task, title, "✅" if r else ("not attempted" if r is None else "❌")])
    table = pd.DataFrame(rows, columns=["Task", "Topic", "Result"])
    print(table.to_string(index=False))
    print(f"\nSQL score: {sum(1 for r in RESULTS.values() if r)} / {len(TASKS)}")

print(f"SQL setup done: {len(TASKS)} tasks. Connected as:", run_query("SELECT CURRENT_USER AS u").u[0])

SQL setup done: 10 tasks. Connected as: intern_readonly


In [2]:
q("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA='week06' ORDER BY TABLE_NAME")

,TABLE_NAME
0,beijing_pek_2014
1,brasilia_bsb_2014
2,capetown_cpt_2014
3,delhi_del_2014
4,london_2014
5,moscow_svo_2014
6,who_pop_tb_all
7,who_pop_tb_some


In [3]:
SQL_01 = """
SELECT
    TABLE_NAME AS table_name,
    COUNT(*) AS n_columns
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'week06'
GROUP BY TABLE_NAME
ORDER BY table_name ASC
"""

answer("s01", SQL_01)

,table_name,n_columns
0,beijing_pek_2014,23
1,brasilia_bsb_2014,23
2,capetown_cpt_2014,23
3,delhi_del_2014,23
4,london_2014,23
5,moscow_svo_2014,23
6,who_pop_tb_all,3
7,who_pop_tb_some,3


✅ s01: correct!


In [4]:
SQL_02 = """
SELECT
    SUM(
        CASE WHEN [ Events] IS NULL THEN 1 ELSE 0 END
    ) AS no_event_days,

    SUM(
        CASE WHEN [ Events] LIKE '%Rain%' THEN 1 ELSE 0 END
    ) AS rain_days,

    SUM(
        CASE
            WHEN [ Events] IS NULL
             AND Precipitationmm > 0
            THEN 1
            ELSE 0
        END
    ) AS wet_but_no_event_days

FROM week06.london_2014
"""

answer("s02", SQL_02)

,no_event_days,rain_days,wet_but_no_event_days
0,114,238,6


✅ s02: correct!


In [5]:
SQL_03 = """
SELECT
    Contract_Type,
    COUNT(*) AS customers,
    SUM(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) AS churned,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 1 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        1
    ) AS churn_rate_pct
FROM week16.customer_churn_prediction
GROUP BY Contract_Type
ORDER BY churn_rate_pct DESC
"""

answer("s03", SQL_03)

,Contract_Type,customers,churned,churn_rate_pct
0,Month-to-Month,158,91,57.6
1,One Year,165,89,53.9
2,Two Year,177,48,27.1


✅ s03: correct!


In [3]:
q("SELECT TOP 2 * FROM week09.comtrade_milk_uk_monthly_14")

,Classification,Year,Period,Period Desc.,Aggregate Level,Is Leaf Code,Trade Flow Code,Trade Flow,Reporter Code,Reporter,...,Qty,Alt Qty Unit Code,Alt Qty Unit,Alt Qty,Netweight (kg),Gross weight (kg),Trade Value (US$),CIF Trade Value (US$),FOB Trade Value (US$),Flag
0,HS,2014,201401,January 2014,4,0,1,Imports,826,United Kingdom,...,None,None,None,None,22404316,None,21950747,None,None,0
1,HS,2014,201401,January 2014,4,0,2,Exports,826,United Kingdom,...,None,None,None,None,60497363,None,46923551,None,None,0


In [10]:
SQL_04 = """
SELECT
    Partner,
    SUM([Trade Value (US$)]) AS total_exports_usd
FROM week09.comtrade_milk_uk_monthly_14
WHERE Partner <> 'World'
  AND [Trade Flow] = 'Exports'
GROUP BY Partner
HAVING COUNT(DISTINCT Period) = 12
ORDER BY total_exports_usd DESC
"""

answer("s04", SQL_04)

,Partner,total_exports_usd
0,Ireland,421228441
1,Algeria,93753841
2,Netherlands,68649082
3,Germany,36096633
4,France,34622982
5,Belgium,29343007
6,China,27317861
7,"China, Hong Kong SAR",19847947
8,Nigeria,12932371
9,Guinea,12018176


✅ s04: correct!


In [11]:
SQL_05 = """
SELECT TOP 5
    who.Country,
    ROUND(
        who.[TB deaths] * 100000.0
        / NULLIF(CAST(wb.[SP.POP.TOTL] AS FLOAT), 0),
        1
    ) AS tb_deaths_per_100k
FROM week06.who_pop_tb_all AS who
INNER JOIN week08.wb_pop_2013 AS wb
    ON who.Country = wb.country
WHERE who.[TB deaths] IS NOT NULL
  AND wb.[SP.POP.TOTL] IS NOT NULL
  AND wb.[SP.POP.TOTL] > 0
ORDER BY tb_deaths_per_100k DESC
"""

answer("s05", SQL_05)

,Country,tb_deaths_per_100k
0,Djibouti,100.6
1,Nigeria,92.6
2,Swaziland,88.0
3,Timor-Leste,83.9
4,Somalia,75.0


✅ s05: correct!


In [12]:
SQL_06 = """
SELECT
    COUNT(*) AS who_countries_without_wb_match
FROM week06.who_pop_tb_all AS who
WHERE NOT EXISTS (
    SELECT 1
    FROM week08.wb_pop_2013 AS wb
    WHERE LTRIM(RTRIM(wb.country))
        = LTRIM(RTRIM(who.Country))
)
"""

answer("s06", SQL_06)

,who_countries_without_wb_match
0,28


✅ s06: correct!


In [3]:
SQL_07 = """
WITH all_weather AS (
    SELECT 'Beijing' AS city,
           CAST([Mean TemperatureC] AS DECIMAL(10, 2)) AS mean_temp
    FROM week06.beijing_pek_2014

    UNION ALL

    SELECT 'Brasilia',
           CAST([Mean TemperatureC] AS DECIMAL(10, 2))
    FROM week06.brasilia_bsb_2014

    UNION ALL

    SELECT 'Cape Town',
           CAST([Mean TemperatureC] AS DECIMAL(10, 2))
    FROM week06.capetown_cpt_2014

    UNION ALL

    SELECT 'Delhi',
           CAST([Mean TemperatureC] AS DECIMAL(10, 2))
    FROM week06.delhi_del_2014

    UNION ALL

    SELECT 'London',
           CAST([Mean TemperatureC] AS DECIMAL(10, 2))
    FROM week06.london_2014

    UNION ALL

    SELECT 'Moscow',
           CAST([Mean TemperatureC] AS DECIMAL(10, 2))
    FROM week06.moscow_svo_2014
)
SELECT
    city,
    ROUND(AVG(mean_temp), 1) AS avg_mean_temp_c
FROM all_weather
GROUP BY city
ORDER BY city ASC
"""

answer("s07", SQL_07)

,city,avg_mean_temp_c
0,Beijing,13.4
1,Brasilia,22.9
2,Cape Town,17.6
3,Delhi,13.7
4,London,12.3
5,Moscow,6.0


✅ s07: correct!


In [3]:
SQL_08 = """
WITH london_days AS (
    SELECT
        MONTH(TRY_CONVERT(date, GMT)) AS month_no,
        TRY_CONVERT(date, GMT) AS day_date,
        [Max TemperatureC] AS max_temp
    FROM week06.london_2014
    WHERE TRY_CONVERT(date, GMT) IS NOT NULL
      AND [Max TemperatureC] IS NOT NULL
),
ranked_days AS (
    SELECT
        month_no,
        day_date,
        max_temp,
        ROW_NUMBER() OVER (
            PARTITION BY month_no
            ORDER BY max_temp DESC, day_date ASC
        ) AS rn
    FROM london_days
)
SELECT
    month_no,
    day_date AS [day],
    max_temp
FROM ranked_days
WHERE rn = 1
ORDER BY month_no ASC
"""

answer("s08", SQL_08)

,month_no,day,max_temp
0,1,2014-01-05,12
1,2,2014-02-24,14
2,3,2014-03-09,19
3,4,2014-04-01,19
4,5,2014-05-19,26
5,6,2014-06-09,26
6,7,2014-07-18,30
7,8,2014-08-06,26
8,9,2014-09-18,26
9,10,2014-10-31,23


✅ s08: correct!


In [5]:
SQL_09 = """
WITH monthly_exports AS (
    SELECT
        Period,
        SUM([Trade Value (US$)]) AS exports_usd
    FROM week09.comtrade_milk_uk_monthly_14
    WHERE Partner <> 'World'
      AND [Trade Flow] = 'Exports'
    GROUP BY Period
)
SELECT
    Period,
    exports_usd,
    SUM(exports_usd) OVER (
        ORDER BY Period
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS ytd_exports_usd
FROM monthly_exports
ORDER BY Period ASC
"""

answer("s09", SQL_09)

,Period,exports_usd,ytd_exports_usd
0,201401,87138654,87138654
1,201402,72489716,159628370
2,201403,86781424,246409794
3,201404,95195778,341605572
4,201405,96201135,437806707
5,201406,99315900,537122607
6,201407,72878716,610001323
7,201408,61737703,671739026
8,201409,60600293,732339319
9,201410,57003417,789342736


✅ s09: correct!


In [5]:
SQL_10 = """
SELECT
    Property_Area,
    Credit_History,
    COUNT(*) AS applications,
    ROUND(
        100.0 * SUM(
            CASE WHEN Loan_Approved = 1 THEN 1 ELSE 0 END
        ) / NULLIF(COUNT(*), 0),
        1
    ) AS approval_rate_pct
FROM week17.loan_approval_prediction
GROUP BY
    Property_Area,
    Credit_History
ORDER BY
    Property_Area ASC,
    Credit_History ASC
"""

answer("s10", SQL_10)

,Property_Area,Credit_History,applications,approval_rate_pct
0,Rural,0,166,0.0
1,Semiurban,1,167,100.0
2,Urban,1,167,100.0


✅ s10: correct!
